# Re-execução SAINT (fiel) + FT-Entmax (bisseção corrigida) — Kaggle

**Motivação (auditoria 2026-09-18):**
- `FTTransformer_entmax`: a bisseção do entmax usava intervalo errado — a saída não era entmax-1,5. Corrigido em `entmax_attention.py` (Jacobiana exata; testes de regressão).
- `SAINTColnorm`: a atenção inter-instâncias só operava sobre o CLS. Reimplementado fiel a Somepalli et al. (2021): embedding FC+ReLU por atributo, MISA sobre a linha inteira (p+1)·d, LN nos resíduos, GELU, cabeça MLP no CLS.

**Fases (cada uma grava um JSON próprio em `/kaggle/working`; nada sobrescreve os canônicos aqui):**
1. Tier 1 (10 datasets × 30 seeds, GridSearchCV)
2. Tier 2 (6 datasets × 30 seeds, N=2000, GridSearchCV)
3. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 novo)
4. Ablação A — **os seis Transformers**, por transferência dos hiperparâmetros do Tier 1 (`run_ablation_a_scaling.py --transformers-only`, 20 seeds). Corrige a assimetria descoberta na auditoria: a versão publicada re-tunou os Transformers em N=2000 enquanto os LSSVMs foram transferidos.
5. Ablações B + C (TWS_5f/TWM_5f/TWC_5f + MKE/MKM/MKH, 30 seeds)
6. Benchmark de escalabilidade — Tabela 19 (`run_table19_benchmark.py`: SAINT mini/full-batch + FT-Entmax)

**Depois:** baixe os JSONs e rode `scripts/merge_rerun_results.py` localmente (ver `docs/rerun_saint_entmax.md`).

**Antes de rodar:** as correções estão na branch `revisao/estatistica-e-proveniencia` (commit `aac573c` ou posterior); a célula 2 faz o checkout dela. Settings → Accelerator → GPU T4.
Se a sessão cair, suba os JSONs parciais como Dataset e aponte `RESUME_DIR`.

In [ ]:
# ── 1. GPU ──
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── 2. Repositório ──
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'   # branch com as correções (não é a main!)
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git branch --show-current
!git log --oneline -3
# Sanidade: o clone precisa conter as correções
import sys; sys.path.insert(0, '.')
from src.models.transformers.sparse_attention.entmax_attention import _EntmaxBisectFunction  # noqa
from src.models.ft_transformer_model import SAINTStage  # noqa
print('correções presentes: entmax (Function) + SAINTStage OK')

In [ ]:
# ── 3. Dependências e dados ──
!pip install -q einops scikit-posthocs openpyxl
!python scripts/download_data.py --tier 1
from src.data.loaders import DatasetLoader
for ds in ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC','ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO']:
    X, y, _ = DatasetLoader.load(ds); print(f'{ds:<9} N={len(y):>6} p={X.shape[1]}')

In [ ]:
# ── 4. Configuração ──
import shutil, json
from pathlib import Path
MODELS = ['SAINTColnorm', 'FTTransformer_entmax']
MODELS_STR = ' '.join(MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2 = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
SEEDS30 = ' '.join(map(str, range(30)))
SEEDS20 = ' '.join(map(str, range(20)))
OUT = {
    'tier1':  'results/rerun_saint_entmax_tier1.json',
    'tier2':  'results/rerun_saint_entmax_tier2.json',
    'n5000':  'results/rerun_saint_entmax_n5000.json',
    'ablA':   'results/rerun_saint_entmax_ablA.json',
    'ablBC':  'results/rerun_saint_entmax_ablBC.json',
    'scal':   'results/rerun_saint_entmax_table19.json',
}
Path('results').mkdir(exist_ok=True)
# Resume: suba os JSONs parciais como Dataset do Kaggle e aponte o diretório
RESUME_DIR = None   # ex.: Path('/kaggle/input/rerun-saint-entmax')
if RESUME_DIR:
    for k, v in OUT.items():
        src = Path(RESUME_DIR) / Path(v).name
        if src.exists():
            shutil.copy(src, v); print('restaurado', v, len(json.load(open(v))))
def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

In [ ]:
# ── 5. Tier 1 ──
!python -u scripts/run_tier1_gridcv.py --models {MODELS_STR} --datasets {TIER1} --seeds {SEEDS30} --output {OUT['tier1']} 2>&1 | tail -n 5
save('tier1')

In [ ]:
# ── 6. Tier 2 (N=2000) ──
!python -u scripts/run_tier2_gridcv.py --models {MODELS_STR} --datasets {TIER2} --seeds {SEEDS30} --n-train 2000 --output {OUT['tier2']} 2>&1 | tail -n 5
save('tier2')

In [ ]:
# ── 7. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 NOVO) ──
# Não usar extract_tier2_fixed_params.py aqui: ele lê TODOS os results/tier2_transformers*.json
# (dumps antigos incluídos) e a deduplicação manteria os best_params da versão antiga.
import collections
recs = [r for r in json.load(open(OUT['tier2'])) if r.get('status') == 'ok' and r.get('best_params')]
by = collections.defaultdict(list)
for r in recs:
    by[(r['variant'], r['dataset'])].append(tuple(sorted(r['best_params'].items())))
cfg = json.load(open('config/tier2_fixed_params.json'))   # mantém as demais variantes
for (v, d), vals in sorted(by.items()):
    mode, cnt = collections.Counter(vals).most_common(1)[0]
    cfg.setdefault(v, {})[d] = dict(mode)
    print(f'{v:<22} {d:<9} {dict(mode)}  [moda {cnt}/{len(vals)}]')
Path('config/tier2_fixed_params_rerun.json').write_text(json.dumps(cfg, indent=2, sort_keys=True))
shutil.copy('config/tier2_fixed_params_rerun.json', '/kaggle/working/tier2_fixed_params_rerun.json')
!python -u scripts/run_tier2_fixedparams.py --models {MODELS_STR} --datasets {TIER2} --seeds {SEEDS30} --n-train 5000 --config config/tier2_fixed_params_rerun.json --output {OUT['n5000']} 2>&1 | tail -n 5
save('n5000')

In [ ]:
# ── 8. Ablação A — SEIS Transformers por transferência do Tier 1 (protocolo dos LSSVMs) ──
# tier1 de referência = tier1_gridcv.json do repo com SAINT/entmax substituídos pelo rerun
!python scripts/merge_rerun_results.py --target results/tier1_gridcv.json --source {OUT['tier1']} --variants {MODELS_STR} --tag kaggle
ALL_TR = 'FTTransformer_softmax FTTransformer_topk FTTransformer_entmax FTTransformer_sparsemax FTTransformerCURColnorm SAINTColnorm'
!python -u scripts/run_ablation_a_scaling.py --transformers-only --tier1 results/tier1_gridcv.json --seeds {SEEDS20} --output {OUT['ablA']} 2>&1 | tail -n 5
save('ablA')
# checagem: todos os registros devem ter protocol == transfer_from_tier1
recs = json.load(open(OUT['ablA']))
print(collections.Counter((r['variant'], r.get('protocol')) for r in recs if r.get('status') == 'ok'))

In [ ]:
# ── 9. Ablações B + C (30 seeds) ──
!python -u scripts/run_tier1_gridcv.py --models {MODELS_STR} --datasets TWS_5f TWM_5f TWC_5f MKE MKM MKH --seeds {SEEDS30} --output {OUT['ablBC']} 2>&1 | tail -n 5
save('ablBC')

In [ ]:
# ── 10. Benchmark de escalabilidade (Tabela 19) ──
# A tabela publicada vem de scripts/run_table19_benchmark.py (fit e predição medidos
# separadamente; formato results/table19_results.json), NÃO de run_transformer_scaling.py.
!python -u scripts/run_table19_benchmark.py --variants SAINT_minibatch,SAINT_fullbatch,FTTransformer_entmax --repeats 3 --output {OUT['scal']} 2>&1 | tail -n 15
save('scal')

In [ ]:
# ── 11. Resumo ──
import statistics as st
from collections import defaultdict
for key in ['tier1', 'tier2', 'n5000', 'ablA', 'ablBC']:
    p = Path(OUT[key])
    if not p.exists(): print(key, 'ausente'); continue
    ok = [r for r in json.load(open(p)) if r.get('status') == 'ok']
    f1 = defaultdict(list)
    for r in ok: f1[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
    print(f'\n=== {key}: {len(ok)} registros ok ===')
    for (v, d), vals in sorted(f1.items()):
        print(f'  {v:<22} {d:<9} {st.mean(vals):.4f} ± {st.pstdev(vals):.4f}  n={len(vals)}')